# Precipitation Data Cleaning – Atlantic Canada

This notebook prepares raw monthly precipitation climate observations from
Environment and Climate Change Canada (ECCC) for analysis.

Scope:
- Region: Atlantic Canada (NS, NB, PE, NL)
- Period: 1995–2025
- Granularity: Monthly
- Core metric: Total Monthly Precipitation (mm)

Output:
- Fact_Precipitation table for Power BI integration

In [3]:
# Libraries used for data wrangling
import pandas as pd
import numpy as np
from pathlib import Path

Raw precipitation data is sourced from the ECCC Climate Observations (monthly CSV files).


The climate station list is used only to define the valid universe of
Atlantic Canada stations.

At this stage, the notebook focuses on structuring and documenting the
data preparation steps, not on final analysis.

In [4]:
# Path to the raw precipitation dataset (to be generated by the download notebook)
raw_path = Path("../Capstone/raw data/precipitation_monthly_raw_atlantic.csv")

precip_raw = pd.read_csv(raw_path)

Initial inspection of the raw precipitation dataset will be performed to:
- Understand column structure
- Identify missing values
- Validate temporal coverage
- Confirm station coverage

In [5]:
precip_raw.info()
precip_raw.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63902 entries, 0 to 63901
Data columns (total 28 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Station Name  63902 non-null  object 
 1   Longitude     63902 non-null  float64
 2   Latitude      63902 non-null  float64
 3   Climate ID    63902 non-null  object 
 4   Province      63902 non-null  object 
 5   Year          63902 non-null  int64  
 6   Month         63902 non-null  int64  
 7   Tm            61298 non-null  float64
 8   DwTm          61298 non-null  float64
 9   D             23072 non-null  float64
 10  Tx            61330 non-null  float64
 11  DwTx          61330 non-null  float64
 12  Tn            61351 non-null  float64
 13  DwTn          61351 non-null  float64
 14  S             35381 non-null  float64
 15  DwS           35381 non-null  float64
 16  S%N           13962 non-null  object 
 17  P             57986 non-null  float64
 18  DwP           57986 non-nu

,Station Name,Longitude,Latitude,Climate ID,Province,Year,Month,Tm,DwTm,D,...,DwP,P%N,S_G,Pd,BS,DwBS,BS%,HDD,CDD,province
0,AMHERST (AUT),-64.267,45.85,8200091,NS,1995,1,-5.2,5.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,603.9,0.0,NS
1,AMHERST (AUT),-64.267,45.85,8200091,NS,1995,2,-8.4,4.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,632.7,0.0,NS
2,AMHERST (AUT),-64.267,45.85,8200091,NS,1995,3,-2.6,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,515.4,0.0,NS
3,AMHERST (AUT),-64.267,45.85,8200091,NS,1995,4,3.2,6.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,354.3,0.0,NS
4,AMHERST (AUT),-64.267,45.85,8200091,NS,1995,5,9.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,260.7,0.0,NS


For precipitation analysis, only the fields required for the fact table
will be retained. At this stage, the expected core fields include:

- Climate ID
- Year
- Month
- Total Precipitation (mm)

In [6]:
precip = precip_raw[
    [
        "Climate ID",
        "Station Name",
        "Province",
        "Year",
        "Month",
        "P"   # Total Monthly Precipitation (mm)
    ]
].copy()

In [7]:
precip.rename(
    columns={
        "Climate ID": "ClimateID",
        "Station Name": "StationName",
        "P": "Precipitation_mm"
    },
    inplace=True
)

Data types will be standardized to ensure consistency across the dataset:
- Year and Month as integers
- Precipitation values as numeric
Invalid or non-numeric values will be coerced and handled explicitly.

In [8]:
precip["Year"] = precip["Year"].astype(int)
precip["Month"] = precip["Month"].astype(int)
precip["Precipitation_mm"] = pd.to_numeric(
    precip["Precipitation_mm"], errors="coerce"
)

In [15]:
precip["Date"] = pd.to_datetime(
    precip["Year"].astype(str) + "-" +
    precip["Month"].astype(str) + "-01"
)

Missing values will be assessed and handled based on data quality rules.
At this stage, records with missing precipitation or incomplete temporal
information are expected to be removed.

In [10]:
precip = precip.dropna(subset=["Precipitation_mm"])
precip = precip[precip["Precipitation_mm"] >= 0]

Duplicate records will be identified and removed to ensure that each
station-month combination appears only once in the dataset.

In [16]:
precip = precip.drop_duplicates(
    subset=["ClimateID", "Date"]
)

In [12]:
precip.describe()

,Year,Month,Precipitation_mm
count,57986.000000,57986.000000,57986.000000
mean,2007.262960,6.460473,96.715474
std,8.914423,3.454694,52.939203
min,1995.000000,1.000000,0.000000
25%,1999.000000,3.000000,60.300000
50%,2006.000000,6.000000,89.800000
75%,2015.000000,9.000000,125.500000
max,2025.000000,12.000000,582.200000


Potential outliers in precipitation values will be inspected using
summary statistics.

Extreme values will not be removed at this stage, as they may represent
valid climatic events. Final decisions on outliers will be addressed
during the analytical phase if required.

In [13]:
precip.sort_values("Precipitation_mm", ascending=False).head(10)

,ClimateID,StationName,Province,Year,Month,Precipitation_mm
52435,8403389,ST. ANTHONY A,NL,2014,3,582.2
10838,8204161,NORTH MOUNTAIN CS,NS,2010,12,530.7
52085,8403255,SAGONA ISLAND,NL,1997,1,516.0
15399,8205601,STILLWATER SHERBROOKE,NS,1996,9,505.2
18701,8206450,WRECK COVE BROOK,NS,2010,12,491.4
6585,8202502,INGONISH BEACH RCS,NS,2010,12,484.8
18107,8206250,WESTPHAL,NS,1996,9,473.9
17840,8206240,WESTERN HEAD,NS,2005,5,453.9
15781,8205700,SYDNEY A,NS,2010,12,442.5
57816,8404343,WRECKHOUSE,NL,2011,10,441.8


A standardized date field will be created using the Year and Month fields
to support time-based analysis and integration with the calendar dimension.

The cleaned precipitation dataset will be saved as a structured fact table
to be used in the Power BI model.

This dataset will form the `Fact_Precipitation` table in the star schema,
linked to:
- Dim_Station (via Climate ID)
- Dim_Calendar (via Date)

In [20]:
fact_precip = precip[
    [
        "ClimateID",
        "Date",
        "Province",
        "Precipitation_mm"
    ]
].copy()

In [21]:
fact_precip.info()
fact_precip.head

<class 'pandas.core.frame.DataFrame'>
Index: 57986 entries, 5 to 63901
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ClimateID         57986 non-null  object        
 1   Date              57986 non-null  datetime64[ns]
 2   Province          57986 non-null  object        
 3   Precipitation_mm  57986 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 2.2+ MB


<bound method NDFrame.head of       ClimateID       Date Province  Precipitation_mm
5       8200091 1995-06-01       NS              61.0
6       8200091 1995-07-01       NS             108.4
7       8200091 1995-08-01       NS               0.0
8       8200091 1995-09-01       NS               7.4
9       8200091 1995-10-01       NS              82.4
...         ...        ...      ...               ...
63897   850B5R1 2013-08-01       NL              65.5
63898   850B5R1 2013-09-01       NL             117.5
63899   850B5R1 2013-10-01       NL              90.5
63900   850B5R1 2013-11-01       NL              82.5
63901   850B5R1 2013-12-01       NL              23.0

[57986 rows x 4 columns]>

In [18]:
output_path = Path("../Capstone/clean data/fact_precipitation.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

fact_precip.to_csv(output_path, index=False)

This notebook focuses on data preparation and quality assurance.
No trend analysis or visualization is performed at this stage.

Further transformations and analysis will be conducted after model alignment
and schema validation.